In [ ]:
import torch
import numpy as np

import time

from botorch.models.gp_regression import SingleTaskGP
from botorch.models.model_list_gp_regression import ModelListGP
from botorch.models.transforms.input import Normalize
from botorch.models.transforms.outcome import Standardize

from botorch import fit_gpytorch_mll
from gpytorch.mlls.sum_marginal_log_likelihood import SumMarginalLogLikelihood
from botorch.optim.optimize import optimize_acqf
from botorch.acquisition.multi_objective.logei import qLogNoisyExpectedHypervolumeImprovement
from botorch.utils.multi_objective.box_decompositions.dominated import DominatedPartitioning

tkwargs = {
    "dtype": torch.double, 
    "device": torch.device("cuda" if torch.cuda.is_available() else "cpu")
}

from mlp import MLP

# Load the surrogate model
mdl = MLP().to(**tkwargs)
mdl.load_state_dict(torch.load('models/surrogate.pth', weights_only=True))
mdl.eval();

In [ ]:
def func1(X):
    # Utilize the MLP surrogate to predict viability
    return mdl.predict(X)

def func2(X):
    # Sum the individual concentrations to a total concentration
    return X.sum(axis = 1).reshape(-1, 1)

def multi_objective(X):
    return torch.cat([func1(X), func2(X)], dim=1)

In [ ]:
def initialize_model(train_X, train_Y):
    models = [
        SingleTaskGP(
            train_X, train_Y[..., i:i+1],
            input_transform=Normalize(train_X.shape[-1]),
            outcome_transform=Standardize(m=1)
        ) for i in range(train_Y.shape[-1])
    ]
    model_list = ModelListGP(*models)
    return SumMarginalLogLikelihood(model_list.likelihood, model_list), model_list

def step_mobo(model, train_X, raw_samples, num_restarts, batch_limit, max_iter):
    acq = qLogNoisyExpectedHypervolumeImprovement(model, REFERENCE, train_X, prune_baseline=True)
    candidates, _ = optimize_acqf(
        acq_function=acq,   
        bounds=BOUNDS,
        q=BATCH_SIZE,
        num_restarts=num_restarts,
        raw_samples=raw_samples,
        options={"batch_limit": batch_limit, "maxiter": max_iter},
        sequential = True
    )
    return candidates.detach(), multi_objective(candidates), candidates

def run_bo(init_X, run_id):
    init_Y = multi_objective(init_X)
    mll, model = initialize_model(init_X, init_Y)

    hvs = [DominatedPartitioning(REFERENCE, init_Y).compute_hypervolume().item()]
    times = [0.0]
    candidates_list = [init_X.cpu().tolist()]
    train_X, train_Y = init_X.clone(), init_Y.clone()

    for i in range(1, N_ITER + 1):
        try:
            start = time.time()
            fit_gpytorch_mll(mll)
            new_X, new_Y, candidates = step_mobo(model, train_X, **DEFAULT_PARAMS)

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            train_X = torch.cat([train_X, new_X])
            train_Y = torch.cat([train_Y, new_Y])

            hv = DominatedPartitioning(REFERENCE, train_Y).compute_hypervolume().item()
            hvs.append(hv)
            times.append(time.time() - start)
            candidates_list.append(candidates.cpu().tolist())

            mll, model = initialize_model(train_X, train_Y)

            print(f"[Run {run_id} | Iter {i}/{N_ITER}] HV={hv:.5f} t={times[-1]:.2f}")

        except RuntimeError as e:
            if "CUDA out of memory" in str(e):
                print(f"[Run {run_id} | Iter {i}] CUDA OOM encountered. Skipping to next run.")
                torch.cuda.empty_cache()
                break
            else:
                raise e

    return hvs, times, candidates_list

In [ ]:
def loadData():
    data = np.concatenate([
        np.load('data/real_data.npy'),
        np.load('data/iter1_data.npy'),
        np.load('data/iter2_data.npy')
    ])
    return torch.Tensor(data).to(**tkwargs)

init_X = loadData()[:,:-2]

# Define reference point. 
# First objective is CPA viability - range [0,1] ideally, realistically can dip below 0 and exceed 1.
# Second objective is total concentration - range [0,\infty], realistically up to ~8M
REFERENCE = torch.tensor([-0.1, 0.0], **tkwargs)
# Define bounds for individual concentrations
dim = init_X.shape[-1]
lower = 0
upper = 6
BOUNDS = torch.tensor([[lower] * dim, [upper] * dim], **tkwargs)

BATCH_SIZE = 10
N_ITER = 20  # Updated number of iterations

# Default hyperparameters
DEFAULT_PARAMS = {
    "raw_samples": 256,
    "num_restarts": 10,
    "batch_limit": 5,
    "max_iter": 100
}

hvs, times, candidates_list = run_bo(init_X, run_id = 0)